## Modelisation

Model creation to predict the missing data

#### Key Methodological Advancements :
* **Domain Knowledge Injection :** formulated a pseudo-Aridity Index ($P / (T + 30)$) to explicitly capture evapotranspiration dynamics
* **Temporal Decay Modeling (EWMA) :** replaced Simple Moving Averages with Exponentially Weighted Moving Averages (EWMA) to mathematically reflect the natural decay of soil moisture
* **L1-Regularized XGBoost :** deployed a MultiOutputRegressor powered by XGBoost. We applied strict L1 regularization (reg_alpha=0.1) to induce feature sparsity, countering the multicollinearity introduced by temporal lags and EWMA

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

print("Initializing data processing pipeline...")

# ========== 1. DATA LOADING ==========
df_train = pd.read_csv("data/train.csv")
df_test = pd.read_csv("data/test.csv")

# ========== 2. FEATURE ENGINEERING (Exponential Smoothing & Aridity Index) ==========
def preprocess_and_engineer(df, is_train=True):
    """
    Aggregates daily meteorological data to weekly frequency.
    Implements Exponentially Weighted Moving Averages (EWMA) and a proxy 
    Aridity Index to capture decaying historical weather effects.
    """
    df['month'] = df['date'].str[5:7].astype(int)
    df['week'] = df.groupby('region_id').cumcount() // 7
    
    agg_dict = {
        'prec': 'sum', 
        'humidity': 'mean', 
        'tmp': 'mean',
        'tmp_max': 'max', 
        'tmp_min': 'min', 
        'wind': 'mean',
        'wind_max': 'max', 
        'month': 'last'
    }
    
    if is_train:
        agg_dict['score'] = 'max'
        
    df_weekly = df.groupby(['region_id', 'week']).agg(agg_dict).reset_index()
    
    # domain-specific feature : Thermal amplitude
    df_weekly['tmp_range'] = df_weekly['tmp_max'] - df_weekly['tmp_min']
    
    # domain-specific feature : Pseudo-Aridity Index (Precipitation to Temperature ratio)
    # a constant (30.0) is added to the denominator to prevent division by zero or negative scaling
    df_weekly['aridity_index'] = df_weekly['prec'] / (df_weekly['tmp'] + 30.0)
    
    # standard temporal lags (t-1, t-2)
    lag_features = ['prec', 'tmp', 'humidity']
    for col in lag_features:
        df_weekly[f'{col}_lag1'] = df_weekly.groupby('region_id')[col].shift(1)
        df_weekly[f'{col}_lag2'] = df_weekly.groupby('region_id')[col].shift(2)
        
    # Exponentially Weighted Moving Averages (EWMA) over a 4-week span
    # EWMA assigns exponentially decaying weights to older observations, which is mathematically superior to flat simple moving averages for drought modeling
    df_weekly['prec_ewma_4'] = df_weekly.groupby('region_id')['prec'].transform(lambda x: x.ewm(span=4, adjust=False).mean())
    df_weekly['tmp_ewma_4'] = df_weekly.groupby('region_id')['tmp'].transform(lambda x: x.ewm(span=4, adjust=False).mean())
    df_weekly['humidity_ewma_4'] = df_weekly.groupby('region_id')['humidity'].transform(lambda x: x.ewm(span=4, adjust=False).mean())

    return df_weekly

train_weekly = preprocess_and_engineer(df_train, is_train=True)
test_weekly = preprocess_and_engineer(df_test, is_train=False)

# target shifting for multi-horizon forecasting (t+1 to t+5)
for i in range(1, 6):
    train_weekly[f'target_week_{i}'] = train_weekly.groupby('region_id')['score'].shift(-i)

train_final = train_weekly.dropna()

# ========== 3. MODEL TRAINING (Regularized Gradient Boosting) ==========
features = [
    'prec', 
    'humidity', 
    'tmp', 
    'tmp_max', 
    'tmp_min', 
    'wind', 
    'wind_max', 
    'month', 
    'tmp_range', 
    'aridity_index',
    'prec_lag1', 
    'prec_lag2', 
    'tmp_lag1', 
    'tmp_lag2', 
    'humidity_lag1', 
    'humidity_lag2',
    'prec_ewma_4', 
    'tmp_ewma_4', 
    'humidity_ewma_4'
]

target_cols = ['target_week_1', 'target_week_2', 'target_week_3', 'target_week_4', 'target_week_5']

X_train = train_final[features]
y_train = train_final[target_cols]

print("Training...")

# model configuration employs L1 regularization (reg_alpha) to induce sparsity and reduce reliance on highly correlated temporal features
base_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    random_state=42,
    n_jobs=-1
)

model = MultiOutputRegressor(base_model)
model.fit(X_train, y_train)

# ========== 4. INFERENCE AND SUBMISSION ==========
print("Executing inference on the test set...")

test_last_week = test_weekly.groupby('region_id').tail(1)
X_test = test_last_week[features]

predictions = model.predict(X_test)

submission = pd.DataFrame(predictions, columns=['pred_week1', 'pred_week2', 'pred_week3', 'pred_week4', 'pred_week5'])
submission.insert(0, 'region_id', test_last_week['region_id'].values)

submission.to_csv('submission.csv', index=False)
print("Submission file generated.")

Initializing data processing pipeline...
Training...
Executing inference on the test set...
Submission file generated.


ancient model with xgboost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

print("Initializing data processing pipeline...")

# ========== 1. DATA LOADING ==========
df_train = pd.read_csv("data/train.csv")
df_test = pd.read_csv("data/test.csv")

# ========== 2. FEATURE ENGINEERING (Lags & Rolling Windows) ==========
def preprocess_and_engineer(df, is_train=True):
    """
    Aggregates daily meteorological data into weekly intervals and 
    generates temporal features (lags and rolling windows) to capture 
    cumulative drought effects.
    """
    # extract month to capture annual seasonality
    df['month'] = df['date'].str[5:7].astype(int)
    
    # establish a weekly sequential index per region
    df['week'] = df.groupby('region_id').cumcount() // 7
    
    # define aggregation metrics per feature type
    agg_dict = {
        'prec': 'sum', 
        'humidity': 'mean', 
        'tmp': 'mean',
        'tmp_max': 'max', 
        'tmp_min': 'min', 
        'wind': 'mean',
        'wind_max': 'max', 
        'month': 'last'
    }
    
    if is_train:
        agg_dict['score'] = 'max'
        
    df_weekly = df.groupby(['region_id', 'week']).agg(agg_dict).reset_index()
    
    # calculate thermal amplitude
    df_weekly['tmp_range'] = df_weekly['tmp_max'] - df_weekly['tmp_min']
    
    # generate discrete temporal lags (memory of the past 2 weeks)
    lag_features = ['prec', 'tmp', 'humidity']
    for col in lag_features:
        df_weekly[f'{col}_lag1'] = df_weekly.groupby('region_id')[col].shift(1)
        df_weekly[f'{col}_lag2'] = df_weekly.groupby('region_id')[col].shift(2)
        
    # generate rolling window statistics (4-week cumulative trends)
    # precipitation requires a rolling sum (cumulative deficit), while temperatures require a rolling mean
    df_weekly['prec_roll_sum_4'] = df_weekly.groupby('region_id')['prec'].transform(lambda x: x.rolling(window=4, min_periods=1).sum())
    df_weekly['tmp_roll_mean_4'] = df_weekly.groupby('region_id')['tmp'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
    df_weekly['humidity_roll_mean_4'] = df_weekly.groupby('region_id')['humidity'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())

    return df_weekly

train_weekly = preprocess_and_engineer(df_train, is_train=True)
test_weekly = preprocess_and_engineer(df_test, is_train=False)

# target shifting for multi-step ahead forecasting (t+1 to t+5)
for i in range(1, 6):
    train_weekly[f'target_week_{i}'] = train_weekly.groupby('region_id')['score'].shift(-i)

# drop rows containing NaNs induced by lag generation and target shifting
train_final = train_weekly.dropna()

# ========== 3. MODEL TRAINING (Optimized Gradient Boosting) ==========
features = [
    'prec', 
    'humidity', 
    'tmp', 
    'tmp_max', 
    'tmp_min', 
    'wind', 
    'wind_max', 
    'month', 
    'tmp_range', 
    'prec_lag1', 
    'prec_lag2', 
    'tmp_lag1', 
    'tmp_lag2', 
    'humidity_lag1', 
    'humidity_lag2',
    'prec_roll_sum_4', 
    'tmp_roll_mean_4', 
    'humidity_roll_mean_4'
]

target_cols = ['target_week_1', 'target_week_2', 'target_week_3', 'target_week_4', 'target_week_5']

X_train = train_final[features]
y_train = train_final[target_cols]

print("Training MultiOutputRegressor with XGBoost base estimator...")

# hyperparameters tuned for tabular time-series to mitigate overfitting
base_model = XGBRegressor(
    n_estimators=300,          # increased tree count for finer learning
    learning_rate=0.03,        # reduced learning rate to ensure stable convergence
    max_depth=5,               # constrained tree depth
    min_child_weight=3,        # requires minimum sum of instance weight in a child to split
    subsample=0.8,             # row-wise stochastic sampling
    colsample_bytree=0.8,      # column-wise stochastic sampling
    random_state=42,
    n_jobs=-1
)

model = MultiOutputRegressor(base_model)
model.fit(X_train, y_train)

# ========== 4. INFERENCE AND SUBMISSION ==========
print("Executing inference on the test set...")

# isolate the final week of the test sequence per region to capture the most recent historical context
test_last_week = test_weekly.groupby('region_id').tail(1)
X_test = test_last_week[features]

predictions = model.predict(X_test)

submission = pd.DataFrame(predictions, columns=['pred_week1', 'pred_week2', 'pred_week3', 'pred_week4', 'pred_week5'])
submission.insert(0, 'region_id', test_last_week['region_id'].values)

submission.to_csv('submission_xgboost.csv', index=False)
print("Submission file generated.")

Initializing data processing pipeline...
Training MultiOutputRegressor with XGBoost base estimator...
Executing inference on the test set...
Pipeline execution completed. Submission file generated.
